In [35]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

In [36]:
orders = spark.read.option("multiLine", True) \
    .json("/user/student/e-commerce/raw_data/json_files/*.json")

In [37]:
orders.printSchema()

root
 |-- _id: struct (nullable = true)
 |    |-- $oid: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: long (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- total_value: double (nullable = true)



In [38]:
orders = orders.drop("_id")

In [39]:
orders.show(5)

+-----------+--------------------+----------+--------+--------------+-----------+
|customer_id|               items|order_date|order_id|payment_status|total_value|
+-----------+--------------------+----------+--------+--------------+-----------+
|        426|[{41, 6}, {52, 2}...|2023-11-05|     643|      refunded|     517.12|
|        588|  [{57, 6}, {15, 8}]|2024-11-13|     334|       pending|     316.68|
|         59|[{2, 3}, {14, 5},...|2023-09-08|     611|      refunded|     879.28|
|        509|[{55, 3}, {46, 4}...|2023-01-31|      67|        failed|       null|
|        427|    [{2, 7}, {7, 5}]|2023-05-04|     224|     completed|     258.89|
+-----------+--------------------+----------+--------+--------------+-----------+
only showing top 5 rows



In [40]:
orders.count()

1000

In [41]:
orders = orders.dropDuplicates()

In [42]:
orders.count()

920

In [43]:
orders_exploded = orders.select(
    col("order_id"),
    col("customer_id"),
    col("order_date"),
    col("payment_status"),
    explode(col("items")).alias("item_detail")
)

In [44]:
orders_exploded.show(5)

+--------+-----------+----------+--------------+-----------+
|order_id|customer_id|order_date|payment_status|item_detail|
+--------+-----------+----------+--------------+-----------+
|     459|        465|2024-01-29|     completed|   {55, 10}|
|     459|        465|2024-01-29|     completed|    {13, 6}|
|     459|        465|2024-01-29|     completed|     {4, 8}|
|     459|        465|2024-01-29|     completed|    {67, 6}|
|     866|        333|2024-04-14|     completed|    {21, 4}|
+--------+-----------+----------+--------------+-----------+
only showing top 5 rows



In [45]:
windowSpec = Window.partitionBy("order_id").orderBy("item_detail.product_id")

orders_details = orders_exploded.select(
    concat(col("order_id"), lit("_"), row_number().over(windowSpec)).alias("order_line_id"),
    col("order_id"),
    col("customer_id"),
    col("order_date"),
    col("item_detail.product_id").alias("product_id"),
    col("item_detail.quantity").cast("int").alias("sales_quantity"),
    col("payment_status")
)

In [46]:
orders_details.show(5)

+-------------+--------+-----------+----------+----------+--------------+--------------+
|order_line_id|order_id|customer_id|order_date|product_id|sales_quantity|payment_status|
+-------------+--------+-----------+----------+----------+--------------+--------------+
|         26_1|      26|        145|2024-06-07|        18|            10|     completed|
|         26_2|      26|        145|2024-06-07|        24|             8|     completed|
|         26_3|      26|        145|2024-06-07|        50|             7|     completed|
|         26_4|      26|        145|2024-06-07|        54|            10|     completed|
|         29_1|      29|        501|2023-06-21|         4|             4|        failed|
+-------------+--------+-----------+----------+----------+--------------+--------------+
only showing top 5 rows



In [47]:
products = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/user/student/e-commerce/raw_data/csv_files/products.csv")

In [48]:
products.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



In [49]:
products.show(5)

+----------+--------------------+-------------+------+
|product_id|        product_name|     category| price|
+----------+--------------------+-------------+------+
|         1|Non-Stick Frying Pan|Home & Garden|  86.4|
|         2|           Webcam HD|  Electronics|236.02|
|         3|           Jump Rope|       Sports| 79.23|
|         4|  Scented Candle Set|Home & Garden| 83.78|
|         5|       Puzzle 1000pc|         Toys| 35.54|
+----------+--------------------+-------------+------+
only showing top 5 rows



In [50]:
orders_with_products = orders_details.join(products, "product_id", how="left")

orders_with_products.filter(col("price").isNull()).show()

+----------+-------------+--------+-----------+----------+--------------+--------------+------------+--------+-----+
|product_id|order_line_id|order_id|customer_id|order_date|sales_quantity|payment_status|product_name|category|price|
+----------+-------------+--------+-----------+----------+--------------+--------------+------------+--------+-----+
+----------+-------------+--------+-----------+----------+--------------+--------------+------------+--------+-----+



In [51]:
orders_with_products = orders_with_products.withColumn(
    "sales_per_product_per_order", col("sales_quantity") * col("price"))

orders_with_products.show(5)

+----------+-------------+--------+-----------+----------+--------------+--------------+------------------+-------------+------+---------------------------+
|product_id|order_line_id|order_id|customer_id|order_date|sales_quantity|payment_status|      product_name|     category| price|sales_per_product_per_order|
+----------+-------------+--------+-----------+----------+--------------+--------------+------------------+-------------+------+---------------------------+
|        18|         26_1|      26|        145|2024-06-07|            10|     completed|       Foam Roller|       Sports| 33.45|                      334.5|
|        24|         26_2|      26|        145|2024-06-07|             8|     completed|     Action Figure|         Toys| 33.04|                     264.32|
|        50|         26_3|      26|        145|2024-06-07|             7|     completed|      Wool Sweater|     Clothing|102.68|                     718.76|
|        54|         26_4|      26|        145|2024-06-07|

In [52]:
orders_final = orders_with_products.select(
    "order_line_id", "order_id", "customer_id", "order_date",
    "product_id", "sales_quantity", "sales_per_product_per_order", "payment_status")

In [53]:
orders_final.printSchema()

orders_final.show(5, truncate=False)

root
 |-- order_line_id: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- sales_quantity: integer (nullable = true)
 |-- sales_per_product_per_order: double (nullable = true)
 |-- payment_status: string (nullable = true)



+-------------+--------+-----------+----------+----------+--------------+---------------------------+--------------+
|order_line_id|order_id|customer_id|order_date|product_id|sales_quantity|sales_per_product_per_order|payment_status|
+-------------+--------+-----------+----------+----------+--------------+---------------------------+--------------+
|26_1         |26      |145        |2024-06-07|18        |10            |334.5                      |completed     |
|26_2         |26      |145        |2024-06-07|24        |8             |264.32                     |completed     |
|26_3         |26      |145        |2024-06-07|50        |7             |718.76                     |completed     |
|26_4         |26      |145        |2024-06-07|54        |10            |1591.3                     |completed     |
|29_1         |29      |501        |2023-06-21|4         |4             |335.12                     |failed        |
+-------------+--------+-----------+----------+----------+------

In [54]:
orders_final = orders_final.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

orders_final.printSchema()

root
 |-- order_line_id: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- product_id: long (nullable = true)
 |-- sales_quantity: integer (nullable = true)
 |-- sales_per_product_per_order: double (nullable = true)
 |-- payment_status: string (nullable = true)



In [61]:
windowSpec = Window.orderBy("order_line_id")

fact_sales = orders_final.withColumn(
    "order_item_sk", row_number().over(windowSpec)
).select(
    "order_item_sk",
    "order_line_id",
    "order_id",
    "customer_id",
    "order_date",
    "product_id",
    "sales_quantity",
    col("sales_per_product_per_order").alias("sales_amount"),
    "payment_status"
)

In [62]:
fact_sales.show(5)

2026-09-05 01:53:53,811 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+-------------+--------+-----------+----------+----------+--------------+------------+--------------+
|order_item_sk|order_line_id|order_id|customer_id|order_date|product_id|sales_quantity|sales_amount|payment_status|
+-------------+-------------+--------+-----------+----------+----------+--------------+------------+--------------+
|            1|        100_1|     100|        434|2024-01-21|        28|             8|      226.08|        failed|
|            2|        100_2|     100|        434|2024-01-21|        39|             9|      418.59|        failed|
|            3|        100_3|     100|        434|2024-01-21|        59|             5|      383.05|        failed|
|            4|        101_1|     101|        575|2024-10-24|         9|             6|        74.4|     completed|
|            5|        101_2|     101|        575|2024-10-24|        14|             8|       262.4|     completed|
+-------------+-------------+--------+-----------+----------+----------+

In [63]:
min_date = fact_sales.select(min("order_date")).first()[0]
max_date = fact_sales.select(max("order_date")).first()[0]

dim_date = (
    spark.range(1)
    .select(
        explode(
            sequence(
                lit(min_date),
                lit(max_date),
                expr("interval 1 day")
            )
        ).alias("order_date")
    )
    .select(
        col("order_date").alias("full_date"),
        dayofmonth(col("order_date")).alias("day_of_month"),
        month(col("order_date")).alias("month_number"),
        date_format(col("order_date"), "MMMM").alias("month_name"),
        concat(lit("Q"), quarter(col("order_date"))).alias("quarter"),
        year(col("order_date")).alias("year")
    )
    .orderBy("full_date")
)

In [67]:
dim_date.printSchema()

dim_date.show(5)

root
 |-- full_date: date (nullable = false)
 |-- day_of_month: integer (nullable = false)
 |-- month_number: integer (nullable = false)
 |-- month_name: string (nullable = false)
 |-- quarter: string (nullable = false)
 |-- year: integer (nullable = false)

+----------+------------+------------+----------+-------+----+
| full_date|day_of_month|month_number|month_name|quarter|year|
+----------+------------+------------+----------+-------+----+
|2023-01-04|           4|           1|   January|     Q1|2023|
|2023-01-05|           5|           1|   January|     Q1|2023|
|2023-01-06|           6|           1|   January|     Q1|2023|
|2023-01-07|           7|           1|   January|     Q1|2023|
|2023-01-08|           8|           1|   January|     Q1|2023|
+----------+------------+------------+----------+-------+----+
only showing top 5 rows



In [65]:
fact_sales.coalesce(1).write.mode("overwrite").parquet("/user/student/e-commerce/cleaned_data/orders")

2026-09-05 01:53:59,473 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [66]:
dim_date.coalesce(1).write.mode("overwrite").parquet("/user/student/e-commerce/cleaned_data/dates")